In [6]:
# =========================
# CELL 1: DATA, LOG RETURNS, WINDOWS, RISK LABELS
# =========================

!pip install yfinance -q

import numpy as np
import pandas as pd
import yfinance as yf
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
tf.random.set_seed(42)

print("TF version:", tf.__version__)

# -----------------------
# CONFIG
# -----------------------
START_DATE = "2010-01-01"
END_DATE   = "2025-01-01"

# A few liquid Indian stocks
STOCK_TICKERS = [
    "RELIANCE.NS",
    "TCS.NS",
    "INFY.NS",
    "HDFCBANK.NS",
]

# Major indices as extra context features
INDEX_TICKERS = {
    "^NSEI":  "nifty50",
    "^CNXIT": "nifty_it",
}

WINDOW = 60           # lookback window in days
TEST_SIZE = 0.15
VAL_SIZE  = 0.15      # of total after taking out test

# -----------------------
# Index feature builder
# -----------------------
def build_index_features(start, end):
    frames = []
    for symbol, short_name in INDEX_TICKERS.items():
        print(f"Downloading {symbol} ({short_name})...")
        df_idx = yf.download(symbol, start=start, end=end, progress=False)

        # Only Close
        df_idx = df_idx[["Close"]].copy()
        df_idx.rename(columns={"Close": f"idx_{short_name}_close"}, inplace=True)

        # Index log-return
        df_idx[f"idx_{short_name}_log_ret"] = np.log(
            df_idx[f"idx_{short_name}_close"] / df_idx[f"idx_{short_name}_close"].shift(1)
        )

        frames.append(df_idx)
        print(f"  Index raw shape: {df_idx.shape}")

    # Inner join all index frames on date
    df_all = frames[0]
    for f in frames[1:]:
        df_all = df_all.join(f, how="inner")

    df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_all.dropna(inplace=True)

    print("Combined index features shape:", df_all.shape)
    print("Index feature columns:", list(df_all.columns))

    return df_all

# -----------------------
# Stock feature builder
# -----------------------
def add_stock_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    df_raw columns: Open, High, Low, Close, Volume
    Features:
      - log_ret  : log-return of Close
      - hl_range : (High-Low)/Close
      - oc_range : (Close-Open)/Open
      - vol_chg  : pct change of Volume
    """
    d = df_raw.copy()

    d["log_ret"]  = np.log(d["Close"] / d["Close"].shift(1))
    d["hl_range"] = (d["High"] - d["Low"]) / d["Close"]
    d["oc_range"] = (d["Close"] - d["Open"]) / d["Open"]
    d["vol_chg"]  = d["Volume"].pct_change()

    d.replace([np.inf, -np.inf], np.nan, inplace=True)
    d.dropna(inplace=True)

    return d

# -----------------------
# Make sliding windows
# -----------------------
def make_windows(df: pd.DataFrame,
                 window: int,
                 return_col: str = "log_ret"):
    """
    Given features with log_ret, etc., create:
      X: (num_windows, window, num_features)
      y_ret: next-day log-return (float) for risk label definition
      dates: date of the *prediction day* (the day after the window)
    """
    df = df.copy()
    df.sort_index(inplace=True)
    df.dropna(inplace=True)

    # Next-day return as regression target proxy (we won't use regression, only sign/magnitude)
    log_ret_next = df[return_col].shift(-1)

    # Can't use last row as X (its next return is NaN)
    df_feat = df.iloc[:-1].copy()
    log_ret_next = log_ret_next.iloc[:-1].copy()

    X_list, y_list, date_list = [], [], []

    for i in range(window - 1, len(df_feat)):
        window_df = df_feat.iloc[i - window + 1 : i + 1]
        X_list.append(window_df.values)
        y_list.append(float(log_ret_next.iloc[i]))
        # prediction day = "tomorrow" relative to last row of window
        date_list.append(window_df.index[-1])

    X = np.stack(X_list)
    y_ret = np.array(y_list, dtype=np.float32)
    dates = np.array(date_list)

    return X, y_ret, dates, list(df_feat.columns)

# -----------------------
# Build full dataset
# -----------------------
idx_features = build_index_features(START_DATE, END_DATE)

X_all_list = []
y_ret_all_list = []
dates_all_list = []
feature_names = None

print("\n=== Downloading stocks and joining indices ===\n")

for ticker in STOCK_TICKERS:
    print(f"Downloading {ticker}...")
    df_raw = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
    print("  Raw shape:", df_raw.shape)

    df_raw = df_raw[["Open", "High", "Low", "Close", "Volume"]].copy()
    df_feat = add_stock_features(df_raw)

    # Join stock features with index features
    df_join = df_feat.join(idx_features, how="inner")
    df_join.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_join.dropna(inplace=True)

    print("  Joined stock+indices shape:", df_join.shape)

    X, y_ret, dates, feat_cols = make_windows(df_join, WINDOW, return_col="log_ret")
    print(f"  Windows shape X: {X.shape}, y_ret: {y_ret.shape}")

    X_all_list.append(X)
    y_ret_all_list.append(y_ret)
    dates_all_list.append(dates)

    if feature_names is None:
        feature_names = feat_cols

# Concatenate all stocks
X_all    = np.concatenate(X_all_list, axis=0)
y_ret_all = np.concatenate(y_ret_all_list, axis=0)
dates_all = np.concatenate(dates_all_list, axis=0)

print("\n=== Combined dataset ===")
print("X_all:", X_all.shape, "y_ret_all:", y_ret_all.shape)

# -----------------------
# Shuffle & split
# -----------------------
rng = np.random.default_rng(42)
perm = rng.permutation(len(y_ret_all))

X_all     = X_all[perm]
y_ret_all = y_ret_all[perm]
dates_all = dates_all[perm]

n_total = len(y_ret_all)
n_test  = int(TEST_SIZE * n_total)
n_val   = int(VAL_SIZE * n_total)
n_train = n_total - n_test - n_val

X_train = X_all[:n_train]
y_ret_train = y_ret_all[:n_train]

X_val   = X_all[n_train:n_train + n_val]
y_ret_val = y_ret_all[n_train:n_train + n_val]

X_test  = X_all[n_train + n_val:]
y_ret_test = y_ret_all[n_train + n_val:]

print(f"\nSplit sizes (windows): train={len(y_ret_train)}, val={len(y_ret_val)}, test={len(y_ret_test)}")

# -----------------------
# Risk labels from log-returns
# -----------------------
# Define "big move" using train only (top 10% absolute log-return)
abs_y_train = np.abs(y_ret_train)
big_move_thr = np.percentile(abs_y_train, 90)

def make_risk_labels(y, thr):
    return (np.abs(y) >= thr).astype("float32")

y_risk_train = make_risk_labels(y_ret_train, big_move_thr)
y_risk_val   = make_risk_labels(y_ret_val,   big_move_thr)
y_risk_test  = make_risk_labels(y_ret_test,  big_move_thr)

print(f"\nBig-move threshold |log_ret| (train, 90th pct): {big_move_thr:.6f}")
print("Big-move ratio train/val/test:",
      y_risk_train.mean(), y_risk_val.mean(), y_risk_test.mean())

# reshape for Keras
y_risk_train = y_risk_train.reshape(-1, 1)
y_risk_val   = y_risk_val.reshape(-1, 1)
y_risk_test  = y_risk_test.reshape(-1, 1)

# -----------------------
# Standardize X only (features)
# -----------------------
n_features = X_train.shape[-1]
scaler_X = StandardScaler()

X_train_flat = X_train.reshape(-1, n_features)
scaler_X.fit(X_train_flat)

def scale_X(X):
    flat = X.reshape(-1, n_features)
    flat_s = scaler_X.transform(flat)
    return flat_s.reshape(X.shape)

X_train_s = scale_X(X_train)
X_val_s   = scale_X(X_val)
X_test_s  = scale_X(X_test)

print("\nFeature count:", n_features)
print("Sample X_train_s[0] shape:", X_train_s[0].shape)
print("Risk label distribution (train):", np.mean(y_risk_train), " (1 = big move)")


TF version: 2.19.0
  Index raw shape: (3680, 2)
  Index raw shape: (3392, 2)
Combined index features shape: (3389, 4)
Index feature columns: [('idx_nifty50_close', '^NSEI'), ('idx_nifty50_log_ret', ''), ('idx_nifty_it_close', '^CNXIT'), ('idx_nifty_it_log_ret', '')]

=== Downloading stocks and joining indices ===

  Raw shape: (3700, 5)
  Joined stock+indices shape: (3387, 13)


/tmp/ipython-input-2227065302.py:49: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_idx = yf.download(symbol, start=start, end=end, progress=False)
/tmp/ipython-input-2227065302.py:49: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_idx = yf.download(symbol, start=start, end=end, progress=False)
/tmp/ipython-input-2227065302.py:152: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_raw = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)


  Windows shape X: (3327, 60, 13), y_ret: (3327,)
  Raw shape: (3700, 5)
  Joined stock+indices shape: (3387, 13)


/tmp/ipython-input-2227065302.py:152: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_raw = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)


  Windows shape X: (3327, 60, 13), y_ret: (3327,)
  Raw shape: (3700, 5)
  Joined stock+indices shape: (3387, 13)


/tmp/ipython-input-2227065302.py:152: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_raw = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)


  Windows shape X: (3327, 60, 13), y_ret: (3327,)
  Raw shape: (3700, 5)
  Joined stock+indices shape: (3387, 13)


/tmp/ipython-input-2227065302.py:152: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_raw = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)


  Windows shape X: (3327, 60, 13), y_ret: (3327,)

=== Combined dataset ===
X_all: (13308, 60, 13) y_ret_all: (13308,)

Split sizes (windows): train=9316, val=1996, test=1996

Big-move threshold |log_ret| (train, 90th pct): 0.024699
Big-move ratio train/val/test: 0.10004294 0.10521042 0.10370742

Feature count: 13
Sample X_train_s[0] shape: (60, 13)
Risk label distribution (train): 0.10004294  (1 = big move)


In [7]:
# =========================
# CELL 2: RISK MODELS & TRAINING
# =========================

from tensorflow.keras import layers, Model, callbacks, optimizers, losses, metrics
import tensorflow as tf

T = WINDOW
F = X_train_s.shape[-1]

# -----------------------
# Temporal Attention
# -----------------------
class TemporalAttention(layers.Layer):
    """
    Attention over time axis.
    Input H: (batch, T, d)
    Output: context (batch, d), alpha (batch, T)
    """
    def __init__(self, attn_dim=64, **kwargs):
        super().__init__(**kwargs)
        self.attn_dim = attn_dim

    def build(self, input_shape):
        d = int(input_shape[-1])
        self.W = self.add_weight(
            shape=(d, self.attn_dim), initializer="glorot_uniform",
            trainable=True, name="W"
        )
        self.b = self.add_weight(
            shape=(self.attn_dim,), initializer="zeros",
            trainable=True, name="b"
        )
        self.v = self.add_weight(
            shape=(self.attn_dim, 1), initializer="glorot_uniform",
            trainable=True, name="v"
        )
        super().build(input_shape)

    def call(self, H):
        # H: (batch, T, d)
        u = tf.tanh(tf.tensordot(H, self.W, axes=[[2], [0]]) + self.b)  # (B,T,attn_dim)
        scores = tf.tensordot(u, self.v, axes=[[2], [0]])               # (B,T,1)
        scores = tf.squeeze(scores, axis=-1)                            # (B,T)
        alpha = tf.nn.softmax(scores, axis=1)                           # (B,T)
        alpha_exp = tf.expand_dims(alpha, axis=-1)                      # (B,T,1)
        context = tf.reduce_sum(alpha_exp * H, axis=1)                  # (B,d)
        return context, alpha

# -----------------------
# Baseline: CNN + BiLSTM -> risk only
# -----------------------
def build_baseline_risk(T, F):
    inp = layers.Input(shape=(T, F), name="input_series")

    x = layers.Conv1D(64, 5, padding="same", activation="relu", name="conv_base")(inp)
    x = layers.BatchNormalization(name="bn_base")(x)
    x = layers.Dropout(0.2, name="dropout_base")(x)

    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True), name="bilstm1")(x)
    x = layers.Dropout(0.2, name="dropout_bilstm1")(x)
    x = layers.Bidirectional(layers.LSTM(64), name="bilstm2")(x)

    x = layers.Dense(64, activation="relu", name="dense_shared")(x)
    x = layers.Dropout(0.3, name="dropout_shared")(x)

    risk_out = layers.Dense(1, activation="sigmoid", name="risk_out")(x)

    model = Model(inputs=inp, outputs=risk_out, name="BaselineRisk")
    return model

# -----------------------
# MSF-CAL: multi-scale CNN + BiLSTM + temporal attention -> risk
# -----------------------
def build_msf_cal_risk(T, F):
    inp = layers.Input(shape=(T, F), name="input_series")

    # Multi-scale conv
    c3 = layers.Conv1D(32, 3, padding="same", activation="relu", name="conv_k3")(inp)
    c5 = layers.Conv1D(32, 5, padding="same", activation="relu", name="conv_k5")(inp)
    c7 = layers.Conv1D(32, 7, padding="same", activation="relu", name="conv_k7")(inp)

    x = layers.Concatenate(name="concat_multiscale")([c3, c5, c7])  # (T, 96)
    x = layers.BatchNormalization(name="bn_multiscale")(x)
    x = layers.Dropout(0.25, name="dropout_multiscale")(x)

    # BiLSTM stack
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True), name="bilstm1")(x)
    x = layers.Dropout(0.25, name="dropout_bilstm1")(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True), name="bilstm2")(x)

    # Temporal attention
    context, alpha = TemporalAttention(attn_dim=64, name="temporal_attention")(x)

    h = layers.Dense(64, activation="relu", name="dense1")(context)
    h = layers.Dropout(0.3, name="dropout1")(h)
    h = layers.Dense(32, activation="relu", name="dense2")(h)

    risk_out = layers.Dense(1, activation="sigmoid", name="risk_out")(h)

    model = Model(inputs=inp, outputs=risk_out, name="MSF_CAL_Risk")
    return model

baseline = build_baseline_risk(T, F)
msf_cal  = build_msf_cal_risk(T, F)

baseline.summary()
msf_cal.summary()

# -----------------------
# Compile (pure risk classification)
# -----------------------
loss_fn = losses.BinaryCrossentropy()

baseline.compile(
    optimizer=optimizers.Adam(1e-3),
    loss=loss_fn,
    metrics=[
        metrics.BinaryAccuracy(name="accuracy"),
        metrics.AUC(name="auc")
    ]
)

msf_cal.compile(
    optimizer=optimizers.Adam(1e-3),
    loss=loss_fn,
    metrics=[
        metrics.BinaryAccuracy(name="accuracy"),
        metrics.AUC(name="auc")
    ]
)

# -----------------------
# Class weights for imbalance (big moves are rare)
# -----------------------
pos_ratio = float(y_risk_train.mean())
neg_ratio = 1.0 - pos_ratio

# Give positive class more weight
w_pos = 0.5 / (pos_ratio + 1e-8)
w_neg = 0.5 / (neg_ratio + 1e-8)
class_weight = {0: w_neg, 1: w_pos}

print("\nClass weights:", class_weight)

# -----------------------
# Callbacks (monitor AUC)
# -----------------------
cb_es_base = callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

cb_lr_base = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

cb_es_msf = callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

cb_lr_msf = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

# -----------------------
# Training
# -----------------------
BATCH_SIZE = 64
EPOCHS = 60

print("\n================ TRAINING BASELINE (risk only) ================\n")
hist_base = baseline.fit(
    X_train_s,
    y_risk_train,
    validation_data=(X_val_s, y_risk_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[cb_es_base, cb_lr_base],
    verbose=1
)

print("\n================ TRAINING MSF-CAL (risk only) ================\n")
hist_msf = msf_cal.fit(
    X_train_s,
    y_risk_train,
    validation_data=(X_val_s, y_risk_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[cb_es_msf, cb_lr_msf],
    verbose=1
)


Model: "BaselineRisk"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_series (InputLayer)       │ (None, 60, 13)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_base (Conv1D)              │ (None, 60, 64)         │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_base (BatchNormalization)    │ (None, 60, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_base (Dropout)          │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm1 (Bidirectional)         │ (None, 60, 128)        │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_bilstm1 (Dropout)       │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm2 (Bidirectional)         │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_shared (Dense)            │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_shared (Dropout)        │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ risk_out (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 177,665 (694.00 KB)

 Trainable params: 177,537 (693.50 KB)

 Non-trainable params: 128 (512.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_series        │ (None, 60, 13)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k3 (Conv1D)    │ (None, 60, 32)    │      1,280 │ input_series[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k5 (Conv1D)    │ (None, 60, 32)    │      2,112 │ input_series[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k7 (Conv1D)    │ (None, 60, 32)    │      2,944 │ input_series[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_multiscale   │ (None, 60, 96)    │          0 │ conv_k3[0][0],    │
│ (Concatenate)       │                   │            │ conv_k5[0][0],    │
│                     │                   │            │ conv_k7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_multiscale       │ (None, 60, 96)    │        384 │ concat_multiscal… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_multiscale  │ (None, 60, 96)    │          0 │ bn_multiscale[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm1             │ (None, 60, 128)   │     82,432 │ dropout_multisca… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_bilstm1     │ (None, 60, 128)   │          0 │ bilstm1[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm2             │ (None, 60, 128)   │     98,816 │ dropout_bilstm1[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_attention  │ [(None, 128),     │      8,320 │ bilstm2[0][0]     │
│ (TemporalAttention) │ (None, 60)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense1 (Dense)      │ (None, 64)        │      8,256 │ temporal_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout1 (Dropout)  │ (None, 64)        │          0 │ dense1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense2 (Dense)      │ (None, 32)        │      2,080 │ dropout1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ risk_out (Dense)    │ (None, 1)         │         33 │ dense2[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 206,657 (807.25 KB)

 Trainable params: 206,465 (806.50 KB)

 Non-trainable params: 192 (768.00 B)


Class weights: {0: 0.5555820563172081, 1: 4.997853462615526}

================ TRAINING BASELINE (risk only) ================

Epoch 1/60
146/146 ━━━━━━━━━━━━━━━━━━━━ 52s 255ms/step - accuracy: 0.6606 - auc: 0.6389 - loss: 0.6546 - val_accuracy: 0.6703 - val_auc: 0.6230 - val_loss: 0.6726 - learning_rate: 0.0010
Epoch 2/60
146/146 ━━━━━━━━━━━━━━━━━━━━ 23s 158ms/step - accuracy: 0.7005 - auc: 0.6851 - loss: 0.6306 - val_accuracy: 0.7044 - val_auc: 0.6296 - val_loss: 0.6563 - learning_rate: 0.0010
Epoch 3/60
146/146 ━━━━━━━━━━━━━━━━━━━━ 41s 158ms/step - accuracy: 0.6970 - auc: 0.6865 - loss: 0.6287 - val_accuracy: 0.6157 - val_auc: 0.6406 - val_loss: 0.6803 - learning_rate: 0.0010
Epoch 4/60
146/146 ━━━━━━━━━━━━━━━━━━━━ 42s 162ms/step - accuracy: 0.6560 - auc: 0.6990 - loss: 0.6213 - val_accuracy: 0.6618 - val_auc: 0.6475 - val_loss: 0.6566 - learning_rate: 0.0010
Epoch 5/60
146/146 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.6767 - auc: 0.7150 - loss: 0.6126
Epoch 5: ReduceLROnPla

In [ ]:
# =========================
# CELL 3: RISK EVALUATION
# =========================

import numpy as np

print("=== Test evaluation (risk only) ===")
base_eval = baseline.evaluate(X_test_s, y_risk_test, verbose=0, return_dict=True)
msf_eval  = msf_cal.evaluate(X_test_s,  y_risk_test, verbose=0, return_dict=True)

print("Baseline:", base_eval)
print("MSF-CAL :", msf_eval)

# -----------------------
# Predictions
# -----------------------
y_true = y_risk_test.ravel().astype(int)

base_prob = baseline.predict(X_test_s, verbose=0).ravel()
msf_prob  = msf_cal.predict(X_test_s, verbose=0).ravel()

# Default 0.5 threshold
base_pred = (base_prob >= 0.5).astype(int)
msf_pred  = (msf_prob  >= 0.5).astype(int)

def risk_stats(y_true, y_pred, name):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    print(f"{name:10s} -> precision: {prec:.3f}, recall: {rec:.3f}, F1: {f1:.3f}, "
          f"TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")

print("\n===== THRESHOLD = 0.5 METRICS (TEST) =====")
risk_stats(y_true, base_pred, "Baseline")
risk_stats(y_true, msf_pred,  "MSF-CAL")

# -----------------------
# "Top 10% days" evaluation (portfolio view)
# -----------------------
print("\nBig-move threshold used for labels: |log_ret| >=", big_move_thr)

n_test = len(y_true)
top_k  = max(1, int(0.10 * n_test))   # top 10% days

# Random baseline: pick top_k days randomly
rng = np.random.default_rng(123)
rand_idx = rng.choice(n_test, size=top_k, replace=False)
rand_pred = np.zeros_like(y_true)
rand_pred[rand_idx] = 1

# Top-k by probability for each model
def topk_pred(prob, k):
    idx_sorted = np.argsort(prob)[::-1]   # descending
    idx_flag   = idx_sorted[:k]
    pred = np.zeros_like(prob, dtype=int)
    pred[idx_flag] = 1
    return pred

base_topk = topk_pred(base_prob, top_k)
msf_topk  = topk_pred(msf_prob,  top_k)

print(f"\n===== TOP 10% DAYS METRICS (TEST, k={top_k}) =====")
risk_stats(y_true, rand_pred,  "Random_top10")
risk_stats(y_true, base_topk,  "Baseline_top10")
risk_stats(y_true, msf_topk,   "MSF-CAL_top10")
